### '서민금융한눈에' api 연결

In [16]:
import requests
import pandas as pd
import certifi

service_key = "OOMzi/L4lsQBtwINCMNBqO7LJpepiMcGg5Hnx0rJ5keZ7Zfb/8bYyDgzjsGgPmlvK2afF0Qf+OnsOARuV0IPcQ=="

url = "http://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo"

params = {
    "serviceKey": service_key,
    "pageNo": 1,
    "numOfRows": 300,
    "resultType": "json",
    "likeTrgt": "사업자"
}

response = requests.get(url, params=params, verify=certifi.where())

if response.status_code == 200:
    data = response.json()
    items = data.get("response", {}).get("body", {}).get("items", {}).get("item", [])
    df = pd.DataFrame(items)
    print(df.shape)
else:
    print("❌ API 호출 오류:", response.status_code, response.text)

(300, 47)


In [140]:
import requests
import pandas as pd
import certifi
import time

service_key = "OOMzi/L4lsQBtwINCMNBqO7LJpepiMcGg5Hnx0rJ5keZ7Zfb/8bYyDgzjsGgPmlvK2afF0Qf+OnsOARuV0IPcQ=="
url = "http://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo"

params = {
    "serviceKey": service_key,
    "pageNo": 1,
    "numOfRows": 100,  # 한 페이지 최대값으로 설정
    "resultType": "json",
    "likeTrgt": "사업자"
}

all_items = []

# 1. 첫 페이지 요청
response = requests.get(url, params=params, verify=certifi.where())
if response.status_code != 200:
    raise Exception(f"❌ API 호출 오류: {response.status_code}, {response.text}")

data = response.json()
body = data.get("response", {}).get("body", {})
total_count = body.get("totalCount", 0)
num_of_rows = body.get("numOfRows", 100)

# 2. 전체 페이지 수 계산
total_pages = (total_count // num_of_rows) + (1 if total_count % num_of_rows > 0 else 0)
print(f"전체 건수: {total_count}, 전체 페이지 수: {total_pages}")

# 3. 페이지 반복
for page in range(1, total_pages + 1):
    params["pageNo"] = page
    response = requests.get(url, params=params, verify=certifi.where())
    if response.status_code != 200:
        print(f"❌ {page} 페이지 호출 실패:", response.status_code)
        continue
    data = response.json()
    items = data.get("response", {}).get("body", {}).get("items", {}).get("item", [])
    all_items.extend(items)
    time.sleep(0.2)  # API 과부하 방지

# 4. DataFrame으로 변환
df = pd.DataFrame(all_items)
print("📦 최종 데이터 크기:", df.shape)


전체 건수: 1734, 전체 페이지 수: 18
📦 최종 데이터 크기: (1734, 47)


In [24]:
import requests
import pandas as pd
import certifi
import time

service_key = "OOMzi/L4lsQBtwINCMNBqO7LJpepiMcGg5Hnx0rJ5keZ7Zfb/8bYyDgzjsGgPmlvK2afF0Qf+OnsOARuV0IPcQ=="
url = "http://apis.data.go.kr/1160100/service/GetSmallLoanFinanceInstituteInfoService/getOrdinaryFinanceInfo"

params = {
    "serviceKey": service_key,
    "pageNo": 1,
    "numOfRows": 100,  # 한 페이지 최대값으로 설정
    "resultType": "json",
    "likeTrgt": "소상공인"
}

all_items = []

# 1. 첫 페이지 요청
response = requests.get(url, params=params, verify=certifi.where())
if response.status_code != 200:
    raise Exception(f"❌ API 호출 오류: {response.status_code}, {response.text}")

data = response.json()
body = data.get("response", {}).get("body", {})
total_count = body.get("totalCount", 0)
num_of_rows = body.get("numOfRows", 100)

# 2. 전체 페이지 수 계산
total_pages = (total_count // num_of_rows) + (1 if total_count % num_of_rows > 0 else 0)
print(f"전체 건수: {total_count}, 전체 페이지 수: {total_pages}")

# 3. 페이지 반복
for page in range(1, total_pages + 1):
    params["pageNo"] = page
    response = requests.get(url, params=params, verify=certifi.where())
    if response.status_code != 200:
        print(f"❌ {page} 페이지 호출 실패:", response.status_code)
        continue
    data = response.json()
    items = data.get("response", {}).get("body", {}).get("items", {}).get("item", [])
    all_items.extend(items)
    time.sleep(0.2)  # API 과부하 방지

# 4. DataFrame으로 변환
df2 = pd.DataFrame(all_items)
print("📦 최종 데이터 크기:", df2.shape)


전체 건수: 2197, 전체 페이지 수: 22
📦 최종 데이터 크기: (2197, 47)


In [102]:
final = pd.concat([df, df2], ignore_index=True).sort_values('basYm', ascending=False)
subset = final[final['finPrdNm'] == '보은군 소상공인지원자금']

# 각 열별 고유값 개수 확인
col_uniques = subset.nunique()

# 모든 열의 고유값이 1개면 모든 값이 같음
all_equal = (col_uniques <= 1).all()

print("모든 열 값이 동일한가요? 👉", all_equal)

# 열 중에서 값이 여러 개인 열만 출력
print(col_uniques[col_uniques > 1])

모든 열 값이 동일한가요? 👉 False
basYm           7
snq             2
irt             2
instCtg         2
lnIcdcst        2
hdlInstDtlVw    2
fileWrtDt       7
dtype: int64


In [96]:
subset

,basYm,snq,finPrdNm,lnLmt,irtCtg,irt,maxTotLnTrm,maxDfrmTrm,maxRdptTrm,rdptMthd,...,prdExisYn,cv19Rfrc,prdCtg,prdNm,cv19SuprCtg,cv19SuprCtn,cv19SuprTgtDtlCtn,mgmDln,prdCtg2,fileWrtDt
1810,202506,227,보은군 소상공인지원자금,5000만원,변동금리,은행금리 - 2,3년,-,-,일시상환,...,Y,-,1,대출상품,-,-,-,None,-,202507010700
1961,202505,227,보은군 소상공인지원자금,5000만원,변동금리,은행금리 - 2,3년,-,-,일시상환,...,Y,-,1,대출상품,-,-,-,None,-,202506010700
2112,202504,227,보은군 소상공인지원자금,5000만원,변동금리,은행금리 - 2,3년,-,-,일시상환,...,Y,-,1,대출상품,-,-,-,None,-,202505010700
2263,202503,227,보은군 소상공인지원자금,5000만원,변동금리,은행금리 - 2,3년,-,-,일시상환,...,Y,-,1,대출상품,-,-,-,None,-,202504010700
2414,202502,227,보은군 소상공인지원자금,5000만원,변동금리,은행금리 - 2,3년,-,-,일시상환,...,Y,-,1,대출상품,-,-,-,None,-,202503010700
2566,202501,227,보은군 소상공인지원자금,5000만원,변동금리,은행금리 - 2,3년,-,-,일시상환,...,Y,-,1,대출상품,-,-,-,None,-,202502071556
3921,202110,404,보은군 소상공인지원자금,5000만원,변동금리,은행금리 - 2%,3년,-,-,일시상환,...,Y,None,1,대출상품,None,None,None,None,None,202502010700


basYm : 기준년월
fileWrtDt : 파일작성일시 \
=> 파일이 계속 업뎃되는 과정에서 누적것 같음. \
=> 그래서 basYm 기준으로 내림차순한 뒤, 중복값 제거 ==> final

In [103]:
final = final.drop_duplicates(subset=['finPrdNm', 'ofrInstNm'],keep = 'first')

In [104]:
final[['finPrdNm', 'ofrInstNm']].value_counts()

finPrdNm                ofrInstNm
(2023) 포항시 소상공인 특례보증    경북신용보증재단     1
옥천군 중·저신용 소상공인 특례보증     충북신용보증재단     1
울산광역시 소상공인 경영안정자금       울산신용보증재단     1
울산광역시 북구 소상공인 경영안정자금    울산신용보증재단     1
울산광역시 동구 소상공인 경영안정자금    울산신용보증재단     1
                                    ..
대환대출 지원 특별보증            서울신용보증재단     1
대전광역시 중소기업 경영안정자금 우대보증  대전신용보증재단     1
대전광역시 소상공인경영개선자금 우대보증   대전신용보증재단     1
대전광역시 소상공인 경영개선자금       대전신용보증재단     1
희망플러스 특례보증              신용보증재단중앙회    1
Name: count, Length: 539, dtype: int64

In [108]:
final['basYm'].value_counts().sort_index(ascending=False)

basYm
202506    192
202501      1
202202    275
202201      5
202110     66
Name: count, dtype: int64

In [109]:
# 2022년도 이하 제거(상품이 업뎃 안되었거나 오래된 상품 같아서 삭제)
final = final[final['basYm'].str[:4].astype(int) > 2022]
final['basYm'].value_counts()

basYm
202506    192
202501      1
Name: count, dtype: int64

In [111]:
# 총 상품수 = 193
len(final)

193

In [113]:
final.head()

,basYm,snq,finPrdNm,lnLmt,irtCtg,irt,maxTotLnTrm,maxDfrmTrm,maxRdptTrm,rdptMthd,...,prdExisYn,cv19Rfrc,prdCtg,prdNm,cv19SuprCtg,cv19SuprCtn,cv19SuprTgtDtlCtn,mgmDln,prdCtg2,fileWrtDt
0,202506,4,경상남도 청년 전세자금(경상남도 협약 상품)(경상남도 청년주택 임차보증금 이자지원사업),9000만원,변동금리,은행별 상이,임대차계약기간 이내 최대 2년,0년,0년,일시상환,...,Y,-,1,대출상품,-,-,-,상시,-,202507010700
1838,202506,282,서울시 중소기업 육성자금 지원(특별자금_희망동행자금),10000만원,변동금리,-,5년,1년,4년,균등분할상환,...,Y,-,1,대출상품,-,-,-,상시,-,202507010700
1831,202506,275,"운송업(화물,택시) 및 건설장비 운영업 저금리 상생 특별보증",20000만원,특정금리,은행별 상이,5년,1년,4년,"만기일시상환, 원금균등분할상환",...,Y,-,1,대출상품,-,-,-,한도 소진 시까지,-,202507010700
1832,202506,276,2024년 북구 소상공인 경영안정자금 지원 특례보증(Ⅱ),3000만원,은행금리,은행별 상이,5년,1년,4년,일시상환/분할상환,...,Y,-,1,대출상품,-,-,-,한도 소진 시까지,-,202507010700
1833,202506,277,서울시 중소기업 육성자금 지원(경영안정자금_혁신형도약자금),30000만원,고정금리,3.5%,"2, 5년",1년,4년,"균등분할상환, 일시상환",...,Y,-,1,대출상품,-,-,-,상시,-,202507010700


In [122]:
final['hdlInst']

0                     농협은행, BNK경남은행
1838                       서울신용보증재단
1831                       대구신용보증재단
1832                   아이엠뱅크, 새마을금고
1833                       서울신용보증재단
                   ...             
17                       미소금융 전국 지점
23                       미소금융 전국 지점
24                       미소금융 전국 지점
25      한국법무보호복지공단, 나눔과기쁨, 더불어사는사람들
2498           NH농협은행 경남본부, BNK경남은행
Name: hdlInst, Length: 193, dtype: object

In [128]:
final['prdCtg2'].value_counts()

prdCtg2
-    193
Name: count, dtype: int64

변수명 영어->한글 & 필요한 변수만 불러오기 

In [131]:
# 열 이름 변경
final = final.rename(columns={
    'basYm': '기준년월',
    'finPrdNm': '금융상품명',
    'lnLmt': '대출한도',
    'irtCtg': '금리구분',
    'irt': '금리',
    'rdptMthd': '상환방법',
    'usge': '용도',
    'trgt': '대상',
    'ofrInstNm': '제공기관명',
    'rsdAreaPamtEqltIstm': '거주지역원금균등분할상환',
    'suprTgtDtlCond': '지원대상 상세조건',
    'crdtSc': '신용등급',
    'rfrcCnpl': '문의처 및 연락처',
    'jnMthd': '가입(신청)방법',
    'ovItrYr': '연체이자율(연)',
    'prftAddIrtCond': '우대금리/가산금리 조건',
    'cnpl': '연락처',
    'rltSite': '관련 사이트',
    'tgtFltr': '대상_필터',
    'hdlInstDtlVw': '취급기관_상세보기용',
    'prdExisYn': '상품존재여부',
    'prdNm': '상품명',
    'mgmDln': '운영기한'
})

# 바꾼 열만 선택
columns_to_keep = [
    '기준년월', '금융상품명', '대출한도', '금리구분', '금리', '상환방법', '용도', '대상',
    '제공기관명', '거주지역원금균등분할상환', '지원대상 상세조건', '신용등급', '문의처 및 연락처',
    '가입(신청)방법', '연체이자율(연)', '우대금리/가산금리 조건', '연락처', '관련 사이트',
    '대상_필터', '취급기관_상세보기용', '상품존재여부', '상품명', '운영기한'
]

final = final[columns_to_keep]


In [134]:
final.to_csv("서민금융진흥원_상품정보.csv", index=False)

In [139]:
import pandas as pd
import os
import re
from typing import Union, List

# 🔧 유틸 함수
def is_missing(value: Union[str, List]) -> bool:
    return value is None or (isinstance(value, str) and value.strip() == "-")

def sanitize_filename(s: str) -> str:
    s = re.sub(r"[^\w가-힣]", "", s)
    return s[:30]

# 🧠 하나의 상품 정보를 자연어 문장으로 변환
def convert_row_to_text(data: dict) -> str:
    lines = []

    lines.append(f"[{data['기준년월']}] 기준 정보입니다.")
    lines.append(f"{data['제공기관명']}에서 '{data['금융상품명']}' 상품을 제공합니다.")
    lines.append(f"상품명(내부 코드용)은 '{data['상품명']}'입니다.")
    lines.append(f"운영기한은 {data['운영기한']}까지입니다.")
    lines.append(f"자금용도는 {data['용도']}입니다.")
    lines.append(f"대출 한도는 {data['대출한도']}입니다.")
    lines.append(f"금리 구분은 {data['금리구분']}이며, 금리는 {data['금리']}입니다.")
    lines.append(f"상환 방식은 {data['상환방법']}입니다.")
    lines.append(f"가입 대상은 {data['대상']}입니다.")

    if not is_missing(data.get("지원대상 상세조건")):
        lines.append(f"지원 대상의 세부 조건은 '{data['지원대상 상세조건']}'입니다.")

    if not is_missing(data.get("신용등급")):
        lines.append(f"요구되는 신용등급은 {data['신용등급']}입니다.")

    if not is_missing(data.get("거주지역원금균등분할상환")):
        lines.append(f"거주 지역에 따른 원금균등분할상환 조건은 {data['거주지역원금균등분할상환']}입니다.")

    if not is_missing(data.get("우대금리/가산금리 조건")):
        lines.append(f"우대금리 또는 가산금리는 다음 조건에 따라 적용됩니다: {data['우대금리/가산금리 조건']}.")

    if not is_missing(data.get("연체이자율(연)")):
        lines.append(f"연체 시 적용되는 이자율은 연 {data['연체이자율(연)']}입니다.")

    if not is_missing(data.get("가입(신청)방법")):
        lines.append(f"가입 또는 신청은 '{data['가입(신청)방법']}'을 통해 가능합니다.")

    if not is_missing(data.get("연락처")):
        lines.append(f"추가 문의는 {data['연락처']}로 가능합니다.")

    if not is_missing(data.get("문의처 및 연락처")):
        lines.append(f"문의처 및 담당 부서는 다음과 같습니다: {data['문의처 및 연락처']}.")

    if not is_missing(data.get("관련 사이트")):
        lines.append(f"자세한 정보는 다음 관련 사이트에서도 확인할 수 있습니다: {data['관련 사이트']}.")

    if not is_missing(data.get("대상_필터")):
        lines.append(f"이 상품은 '{data['대상_필터']}'에 해당하는 분들을 위한 상품입니다.")

    if not is_missing(data.get("취급기관_상세보기용")):
        lines.append(f"해당 상품은 '{data['취급기관_상세보기용']}'에서 취급합니다.")

    if not is_missing(data.get("상품존재여부")):
        lines.append(f"해당 상품은 현재 {'존재합니다' if 'Y' in data['상품존재여부'] else '존재하지 않습니다'}.")

    return "\n".join(lines)

# 📁 전체 CSV 처리 → 각 상품을 txt 파일로 저장
def process_csv_to_text(csv_path: str, output_dir: str = "서민금융_금융상품"):
    df = pd.read_csv(csv_path)
    os.makedirs(output_dir, exist_ok=True)

    for idx, row in df.iterrows():
        row_dict = row.to_dict()
        기관명 = sanitize_filename(str(row_dict.get("제공기관명", "unknown")))
        상품명 = sanitize_filename(str(row_dict.get("금융상품명", f"상품_{idx}")))

        filename = f"{str(idx+1).zfill(3)}_{기관명}_{상품명}.txt"
        filepath = os.path.join(output_dir, filename)

        text = convert_row_to_text(row_dict)

        with open(filepath, "w", encoding="utf-8") as out:
            out.write(text)

        print(f"✅ 저장 완료: {filename}")

# ✅ 실행
process_csv_to_text("서민금융진흥원_상품정보.csv")

✅ 저장 완료: 001_BNK경남은행_경상남도청년전세자금경상남도협약상품경상남도청년주택임차보증.txt
✅ 저장 완료: 002_서울신용보증재단_서울시중소기업육성자금지원특별자금_희망동행자금.txt
✅ 저장 완료: 003_대구신용보증재단_운송업화물택시및건설장비운영업저금리상생특별보증.txt
✅ 저장 완료: 004_대구신용보증재단_2024년북구소상공인경영안정자금지원특례보증Ⅱ.txt
✅ 저장 완료: 005_서울신용보증재단_서울시중소기업육성자금지원경영안정자금_혁신형도약자금.txt
✅ 저장 완료: 006_서울신용보증재단_서울시중소기업육성자금지원경영안정자금_이커머스입점피해회복.txt
✅ 저장 완료: 007_서울신용보증재단_서울시중소기업육성자금지원경영안정자금_성장기반자금.txt
✅ 저장 완료: 008_서울신용보증재단_서울시중소기업육성자금지원일반자금_신속드림자금.txt
✅ 저장 완료: 009_서울신용보증재단_서울시중소기업육성자금지원특별자금_안심금리자금20.txt
✅ 저장 완료: 010_서울신용보증재단_서울시중소기업육성자금지원특별자금_친환경기업자금.txt
✅ 저장 완료: 011_IBK기업은행_IBK청년전월세대출전세.txt
✅ 저장 완료: 012_서울신용보증재단_서울시중소기업육성자금_시설자금.txt
✅ 저장 완료: 013_울산신용보증재단_울산광역시저신용소상공인특례보증.txt
✅ 저장 완료: 014_울산신용보증재단_경남은행특별출연소기업소상공인새희망새출발금융지원협약보증.txt
✅ 저장 완료: 015_울산신용보증재단_내생애첫번째맞춤형특례보증.txt
✅ 저장 완료: 016_세종신용보증재단_희망어부바특별보증.txt
✅ 저장 완료: 017_전북신용보증재단_고창군희망더드림특례보증.txt
✅ 저장 완료: 018_전북신용보증재단_완주군희망더드림특례보증.txt
✅ 저장 완료: 019_대구신용보증재단_2024년달성군소상공인경영안정자금지원특례보증.txt
✅ 저장 완료: 020_대구신용보증재단_2024년서구소상공인경영안정자금지원특례보증.txt
✅ 저장 완료: 021_대구신용보증재단_2024년중구소상공인경영안

### 기업마당 크롤링

##### 1페이지만 크롤링

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd


options = Options()
# options.add_argument("--headless")  
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)
url = "https://finlife.fss.or.kr/finlife/ldng/indvlBusi/list.do?menuNo=700072"
driver.get(url)

# 금융상품 검색 클릭
search_btn = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "금융상품 검색")]'))
)
search_btn.click()
print("✅ 검색 버튼 클릭 완료")

# 페이지 끝까지 스크롤 내리기 
prev_height = driver.execute_script('return document.body.scrollHeight')
while True:
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
    time.sleep(5)
    
    curr_height = driver.execute_script('return document.body.scrollHeight')
    if curr_height == prev_height:
        break
    prev_height = curr_height

html = BeautifulSoup(driver.page_source)
txt = html.find_all(class_ = 'onOffTr')
text_list = [i.text for i in txt]

# 크롤링한 데이터 df화 
columns = ['금융회사', '상품명', '자금용도', '가입대상', '대출종류', '금리방식', '상환방식', '전월 평균금리', '대표번호']
cleaned_data = []

for raw in text_list:
    # 줄 단위로 나누고 공백 제거
    lines = [line.strip() for line in raw.strip().split('\n') if line.strip()]

    if len(lines) >= 8:
        # 기본 정보 8개 추출
        base_info = lines[:8]
        
        # 대표번호 찾기: 숫자만 있고 '대표번호' 근처에 있는 값 찾기
        phone_number = ''
        for i, val in enumerate(lines):
            if '대표번호' in val and i > 0:
                candidate = lines[i - 1]
                if candidate.replace('-', '').isdigit():
                    phone_number = candidate
                    break

        # 한 행 완성
        row = base_info + [phone_number]
        cleaned_data.append(row)

# DataFrame으로 변환
df = pd.DataFrame(cleaned_data, columns=columns)

# 출력
print(df.head())

# 저장 (선택)
# df.to_csv("개인사업자대출_상품정보.csv", index=False)

✅ 검색 버튼 클릭 완료
       금융회사                상품명 자금용도  가입대상              대출종류        금리방식  \
0      국민은행     KB 동반성장협약 상생대출   일반  제한없음              담보대출  고정금리, 변동금리   
1  농협은행주식회사             채움 상생론   일반  제한없음  담보대출, 보증대출, 신용대출        변동금리   
2      우리은행  우리CUBE론-X(BIZ프라임)   일반    기타  담보대출, 보증대출, 신용대출  고정금리, 변동금리   
3    중소기업은행       IBK일자리Plus대출   일반    기타  담보대출, 보증대출, 신용대출  고정금리, 변동금리   
4  농협은행주식회사           NH산업단지대출   일반  제한없음              담보대출  고정금리, 변동금리   

                      상환방식 전월 평균금리      대표번호  
0                   만기일시상환   1.04%  15889999  
1  만기일시상환, 원금분할상환, 원리금분할상환   2.58%  16613000  
2     만기일시상환, 원금분할상환, 임의상환   2.64%  15885000  
3  만기일시상환, 원금분할상환, 원리금분할상환   2.77%  15662566  
4  만기일시상환, 원금분할상환, 원리금분할상환   3.14%  16613000  


##### 5페이지까지 크롤링

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time

options = Options()
options.add_argument("--start-maximized")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
url = "https://finlife.fss.or.kr/finlife/ldng/indvlBusi/list.do?menuNo=700072"
driver.get(url)

# "금융상품 검색" 클릭
WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "금융상품 검색")]'))
).click()

time.sleep(2)

columns = ['금융회사', '상품명', '자금용도', '가입대상', '대출종류', '금리방식', '상환방식', '전월 평균금리', '대표번호']
all_data = []

current_page = 1
max_page = 5
all_data = []

while current_page <= max_page:
    # 페이지 끝까지 스크롤 내리기 
    prev_height = driver.execute_script('return document.body.scrollHeight')
    while True:
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
        time.sleep(5)
    
        curr_height = driver.execute_script('return document.body.scrollHeight')
        if curr_height == prev_height:
            break
        prev_height = curr_height

    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    items = soup.find_all(class_="onOffTr")

    for item in items:
        lines = [line.strip() for line in item.text.strip().split('\n') if line.strip()]
        if len(lines) >= 8:
            base_info = lines[:8]
            phone = ''
            for i, val in enumerate(lines):
                if '대표번호' in val and i > 0:
                    candidate = lines[i - 1]
                    if candidate.replace('-', '').isdigit():
                        phone = candidate
                        break
            all_data.append(base_info + [phone])
    print(f"✅ {current_page}페이지 완료")

    # 페이지 클릭 로직 
    if current_page < max_page:
        try:
            next_page = current_page + 1
            next_btn = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable(
                    (By.CSS_SELECTOR, f'a[data-pageindex="{next_page}"]')
                )
            )
            driver.execute_script("arguments[0].click();", next_btn)  # JS로 클릭
            current_page += 1
            time.sleep(2)  
        except Exception as e:
            print(f"❌ {next_page}페이지 클릭 실패: {e}")
            break
    else:
        break


# 데이터프레임 생성
import pandas as pd
columns = ['금융회사', '상품명', '자금용도', '가입대상', '대출종류', '금리방식', '상환방식', '전월 평균금리', '대표번호']
df = pd.DataFrame(all_data, columns=columns)
print(df.head())


✅ 1페이지 완료
✅ 2페이지 완료
✅ 3페이지 완료
✅ 4페이지 완료
✅ 5페이지 완료
       금융회사                상품명 자금용도  가입대상              대출종류        금리방식  \
0      국민은행     KB 동반성장협약 상생대출   일반  제한없음              담보대출  고정금리, 변동금리   
1  농협은행주식회사             채움 상생론   일반  제한없음  담보대출, 보증대출, 신용대출        변동금리   
2      우리은행  우리CUBE론-X(BIZ프라임)   일반    기타  담보대출, 보증대출, 신용대출  고정금리, 변동금리   
3    중소기업은행       IBK일자리Plus대출   일반    기타  담보대출, 보증대출, 신용대출  고정금리, 변동금리   
4  농협은행주식회사           NH산업단지대출   일반  제한없음              담보대출  고정금리, 변동금리   

                      상환방식 전월 평균금리      대표번호  
0                   만기일시상환   1.04%  15889999  
1  만기일시상환, 원금분할상환, 원리금분할상환   2.58%  16613000  
2     만기일시상환, 원금분할상환, 임의상환   2.64%  15885000  
3  만기일시상환, 원금분할상환, 원리금분할상환   2.77%  15662566  
4  만기일시상환, 원금분할상환, 원리금분할상환   3.14%  16613000  


In [34]:
df

,금융회사,상품명,자금용도,가입대상,대출종류,금리방식,상환방식,전월 평균금리,대표번호
0,국민은행,KB 동반성장협약 상생대출,일반,제한없음,담보대출,"고정금리, 변동금리",만기일시상환,1.04%,15889999
1,농협은행주식회사,채움 상생론,일반,제한없음,"담보대출, 보증대출, 신용대출",변동금리,"만기일시상환, 원금분할상환, 원리금분할상환",2.58%,16613000
2,우리은행,우리CUBE론-X(BIZ프라임),일반,기타,"담보대출, 보증대출, 신용대출","고정금리, 변동금리","만기일시상환, 원금분할상환, 임의상환",2.64%,15885000
3,중소기업은행,IBK일자리Plus대출,일반,기타,"담보대출, 보증대출, 신용대출","고정금리, 변동금리","만기일시상환, 원금분할상환, 원리금분할상환",2.77%,15662566
4,농협은행주식회사,NH산업단지대출,일반,제한없음,담보대출,"고정금리, 변동금리","만기일시상환, 원금분할상환, 원리금분할상환",3.14%,16613000
5,주식회사 케이뱅크,사장님 부동산담보대출,"일반,대환",제한없음,담보대출,변동금리,"만기일시상환, 원금분할상환, 임의상환",3.19%,15221000
6,농협은행주식회사,행복채움 프랜차이즈론,일반,기타,"담보대출, 보증대출, 신용대출","고정금리, 변동금리","만기일시상환, 원금분할상환, 원리금분할상환, 임의상환",3.21%,16613000
7,중소기업은행,중기부R&D사업화자금대출,일반,기타,"담보대출, 보증대출, 신용대출","고정금리, 변동금리","만기일시상환, 원금분할상환, 원리금분할상환",3.39%,15662566
8,우리은행,우리CUBE론-X(우리자산신탁),일반,제한없음,"담보대출, 보증대출, 신용대출","고정금리, 변동금리","만기일시상환, 원금분할상환, 임의상환",3.40%,15885000
9,국민은행,KB 우량산업단지기업 우대대출,일반,제한없음,"담보대출, 보증대출","고정금리, 변동금리","만기일시상환, 원금분할상환, 원리금분할상환",3.43%,15889999


##### 모든 페이지 크롤링

In [38]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time

options = Options()
options.add_argument("--start-maximized")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
url = "https://finlife.fss.or.kr/finlife/ldng/indvlBusi/list.do?menuNo=700072"
driver.get(url)

# "금융상품 검색" 클릭
WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "금융상품 검색")]'))
).click()

time.sleep(2)

columns = ['금융회사', '상품명', '자금용도', '가입대상', '대출종류', '금리방식', '상환방식', '전월 평균금리', '대표번호']
all_data = []

current_page = 1
max_page = 72
all_data = []

while current_page <= max_page:
    # 페이지 끝까지 스크롤 내리기 
    prev_height = driver.execute_script('return document.body.scrollHeight')
    while True:
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
        time.sleep(5)
    
        curr_height = driver.execute_script('return document.body.scrollHeight')
        if curr_height == prev_height:
            break
        prev_height = curr_height

    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    items = soup.find_all(class_="onOffTr")

    for item in items:
        lines = [line.strip() for line in item.text.strip().split('\n') if line.strip()]
        if len(lines) >= 8:
            base_info = lines[:8]
            phone = ''
            for i, val in enumerate(lines):
                if '대표번호' in val and i > 0:
                    candidate = lines[i - 1]
                    if candidate.replace('-', '').isdigit():
                        phone = candidate
                        break
            all_data.append(base_info + [phone])
    print(f"✅ {current_page}페이지 완료")

    # 페이지 클릭 로직 
    if current_page < max_page:
        try:
            next_page = current_page + 1
            next_btn = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable(
                    (By.CSS_SELECTOR, f'a[data-pageindex="{next_page}"]')
                )
            )
            driver.execute_script("arguments[0].click();", next_btn)  # JS로 클릭
            current_page += 1
            time.sleep(2)  
        except Exception as e:
            print(f"❌ {next_page}페이지 클릭 실패: {e}")
            break
    else:
        break


# 데이터프레임 생성
import pandas as pd
columns = ['금융회사', '상품명', '자금용도', '가입대상', '대출종류', '금리방식', '상환방식', '전월 평균금리', '대표번호']
df = pd.DataFrame(all_data, columns=columns)
print(df.head())

# 종료
driver.quit()


✅ 1페이지 완료
✅ 2페이지 완료
✅ 3페이지 완료
✅ 4페이지 완료
✅ 5페이지 완료
✅ 6페이지 완료
✅ 7페이지 완료
✅ 8페이지 완료
✅ 9페이지 완료
✅ 10페이지 완료
✅ 11페이지 완료
✅ 12페이지 완료
✅ 13페이지 완료
✅ 14페이지 완료
✅ 15페이지 완료
✅ 16페이지 완료
✅ 17페이지 완료
✅ 18페이지 완료
✅ 19페이지 완료
✅ 20페이지 완료
✅ 21페이지 완료
✅ 22페이지 완료
✅ 23페이지 완료
✅ 24페이지 완료
✅ 25페이지 완료
✅ 26페이지 완료
✅ 27페이지 완료
✅ 28페이지 완료
✅ 29페이지 완료
✅ 30페이지 완료
✅ 31페이지 완료
✅ 32페이지 완료
✅ 33페이지 완료
✅ 34페이지 완료
✅ 35페이지 완료
✅ 36페이지 완료
✅ 37페이지 완료
✅ 38페이지 완료
✅ 39페이지 완료
✅ 40페이지 완료
✅ 41페이지 완료
✅ 42페이지 완료
✅ 43페이지 완료
✅ 44페이지 완료
✅ 45페이지 완료
✅ 46페이지 완료
✅ 47페이지 완료
✅ 48페이지 완료
✅ 49페이지 완료
✅ 50페이지 완료
✅ 51페이지 완료
✅ 52페이지 완료
✅ 53페이지 완료
✅ 54페이지 완료
✅ 55페이지 완료
✅ 56페이지 완료
✅ 57페이지 완료
✅ 58페이지 완료
✅ 59페이지 완료
✅ 60페이지 완료
✅ 61페이지 완료
✅ 62페이지 완료
✅ 63페이지 완료
✅ 64페이지 완료
✅ 65페이지 완료
✅ 66페이지 완료
✅ 67페이지 완료
✅ 68페이지 완료
✅ 69페이지 완료
✅ 70페이지 완료
✅ 71페이지 완료
✅ 72페이지 완료
       금융회사                상품명 자금용도  가입대상              대출종류        금리방식  \
0      국민은행     KB 동반성장협약 상생대출   일반  제한없음              담보대출  고정금리, 변동금리   
1  농협은행주식회사             채움 상생론   일반  제한없음  담보대출, 보증대출, 신용대출        

In [46]:
df

,금융회사,상품명,자금용도,가입대상,대출종류,금리방식,상환방식,전월 평균금리,대표번호
0,국민은행,KB 동반성장협약 상생대출,일반,제한없음,담보대출,"고정금리, 변동금리",만기일시상환,1.04%,15889999
1,농협은행주식회사,채움 상생론,일반,제한없음,"담보대출, 보증대출, 신용대출",변동금리,"만기일시상환, 원금분할상환, 원리금분할상환",2.58%,16613000
2,우리은행,우리CUBE론-X(BIZ프라임),일반,기타,"담보대출, 보증대출, 신용대출","고정금리, 변동금리","만기일시상환, 원금분할상환, 임의상환",2.64%,15885000
3,중소기업은행,IBK일자리Plus대출,일반,기타,"담보대출, 보증대출, 신용대출","고정금리, 변동금리","만기일시상환, 원금분할상환, 원리금분할상환",2.77%,15662566
4,농협은행주식회사,NH산업단지대출,일반,제한없음,담보대출,"고정금리, 변동금리","만기일시상환, 원금분할상환, 원리금분할상환",3.14%,16613000
...,...,...,...,...,...,...,...,...,...
708,KB저축은행,PF대출,"일반,창업,대환",제한없음,담보대출,"고정금리, 변동금리","만기일시상환, 임의상환",-,18990900
709,MS저축은행,부동산담보대출,일반,제한없음,담보대출,"고정금리, 변동금리","만기일시상환, 원금분할상환, 원리금분할상환",-,0537565200
710,MS저축은행,할인어음,일반,제한없음,"담보대출, 신용대출","고정금리, 변동금리",만기일시상환,-,0537565200
711,OK저축은행,숙박업OK론Ⅱ,일반,제한없음,담보대출,고정금리,"만기일시상환, 원금분할상환, 원리금분할상환",-,18997979


In [47]:
# csv로 저장
df.to_csv("기업마당_개인사업자대출.csv", index=False)

#### 상세정보도 크롤링

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time

options = Options()
options.add_argument("--start-maximized")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
url = "https://finlife.fss.or.kr/finlife/ldng/indvlBusi/list.do?menuNo=700072"
driver.get(url)

# "금융상품 검색" 클릭
WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "금융상품 검색")]'))
).click()

time.sleep(2)

columns = ['금융회사', '상품명', '자금용도', '가입대상', '대출종류', '금리방식', '상환방식', '전월 평균금리', '대표번호','상세정보']
all_data = []

current_page = 1
max_page = 2
all_data = []

while current_page <= max_page:
    # 페이지 끝까지 스크롤
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
    time.sleep(2)

    # BeautifulSoup 파싱
    soup = BeautifulSoup(driver.page_source, "html.parser")
    items = soup.find_all(class_="onOffTr")

    for idx, item in enumerate(items):
        # 기본 정보 추출
        lines = [line.strip() for line in item.text.strip().split('\n') if line.strip()]
        if len(lines) < 8:
            continue
        base_info = lines[:8]

        # 대표번호 추출
        phone = ''
        for i, val in enumerate(lines):
            if '대표번호' in val and i > 0:
                candidate = lines[i - 1]
                if candidate.replace('-', '').isdigit():
                    phone = candidate
                    break

        # 상세 버튼 클릭
        try:
            detail_buttons = driver.find_elements(By.CSS_SELECTOR, 'a.tabel-more')
            driver.execute_script("arguments[0].click();", detail_buttons[idx])
            time.sleep(1.5)

            # 상세정보가 포함된 show-box 파싱
            detail_soup = BeautifulSoup(driver.page_source, "html.parser")
            show_boxes = detail_soup.select("tr.show-box")
            detail_text = show_boxes[idx].get_text(separator=' | ', strip=True) if idx < len(show_boxes) else ''

        except Exception as e:
            print(f"❌ 상세 정보 추출 실패 (index {idx}): {e}")
            detail_text = ''

        all_data.append(base_info + [phone, detail_text])
        print(f"✅ {current_page}페이지 - {idx+1}번째 상품 완료")

    # 다음 페이지 클릭
    if current_page < max_page:
        try:
            next_page = current_page + 1
            next_btn = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, f'a[data-pageindex="{next_page}"]'))
            )
            driver.execute_script("arguments[0].click();", next_btn)
            current_page += 1
            time.sleep(2)
        except Exception as e:
            print(f"❌ {next_page}페이지 클릭 실패: {e}")
            break
    else:
        break

# 데이터 저장
df = pd.DataFrame(all_data, columns=columns)
print(df.head())
# 종료
driver.quit()

✅ 1페이지 - 1번째 상품 완료
✅ 1페이지 - 2번째 상품 완료
✅ 1페이지 - 3번째 상품 완료
✅ 1페이지 - 4번째 상품 완료
✅ 1페이지 - 5번째 상품 완료
✅ 1페이지 - 6번째 상품 완료
✅ 1페이지 - 7번째 상품 완료
✅ 1페이지 - 8번째 상품 완료
✅ 1페이지 - 9번째 상품 완료
✅ 1페이지 - 10번째 상품 완료
✅ 2페이지 - 1번째 상품 완료
✅ 2페이지 - 2번째 상품 완료
✅ 2페이지 - 3번째 상품 완료
✅ 2페이지 - 4번째 상품 완료
✅ 2페이지 - 5번째 상품 완료
✅ 2페이지 - 6번째 상품 완료
✅ 2페이지 - 7번째 상품 완료
✅ 2페이지 - 8번째 상품 완료
✅ 2페이지 - 9번째 상품 완료
✅ 2페이지 - 10번째 상품 완료
       금융회사                상품명 자금용도  가입대상              대출종류        금리방식  \
0      국민은행     KB 동반성장협약 상생대출   일반  제한없음              담보대출  고정금리, 변동금리   
1  농협은행주식회사             채움 상생론   일반  제한없음  담보대출, 보증대출, 신용대출        변동금리   
2      우리은행  우리CUBE론-X(BIZ프라임)   일반    기타  담보대출, 보증대출, 신용대출  고정금리, 변동금리   
3    중소기업은행       IBK일자리Plus대출   일반    기타  담보대출, 보증대출, 신용대출  고정금리, 변동금리   
4  농협은행주식회사           NH산업단지대출   일반  제한없음              담보대출  고정금리, 변동금리   

                      상환방식 전월 평균금리      대표번호  \
0                   만기일시상환   1.04%  15889999   
1  만기일시상환, 원금분할상환, 원리금분할상환   2.58%  16613000   
2     만기일시상환, 원금분할상환, 임

In [7]:
df.to_csv("금융상품_상세정보포함.csv", index=False, encoding='utf-8-sig')

##### JSON 파일로 저장(최종)

In [47]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import json
import time

# ✅ Selenium 옵션 설정
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
url = "https://finlife.fss.or.kr/finlife/ldng/indvlBusi/list.do?menuNo=700072"
driver.get(url)

# ✅ "금융상품 검색" 클릭
WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.XPATH, '//button[contains(text(), "금융상품 검색")]'))
).click()
time.sleep(2)

# ✅ 상세항목 파싱 함수
def extract_detailed_info(soup):
    target_keys = [
        "적용가능 금리", "대출한도", "가입대상 세부요건", "가입방법",
        "우대금리", "중도상환 수수료", "대출부대비용", "연체이자율", "담당부서 및 연락처"
    ]
    data = {}

    dl_sets = soup.select("div.dl-set > dl")
    for dl in dl_sets:
        dt_tag = dl.find("dt")
        dd_tags = dl.find_all("dd")

        if not dt_tag or not dd_tags:
            continue

        key = dt_tag.get_text(strip=True).replace('\xa0', ' ').strip()
        if key not in target_keys:
            # 유사 항목 대응 (예: 공백이 다른 경우 등)
            matched = [k for k in target_keys if k in key]
            if matched:
                key = matched[0]
            else:
                continue

        combined_lines = []
        for dd_tag in dd_tags:
            raw_html = dd_tag.decode_contents().replace('\xa0', ' ')
            if "<br>" in raw_html:
                lines = [BeautifulSoup(line, 'html.parser').get_text(strip=True)
                         for line in raw_html.split('<br>') if line.strip()]
                combined_lines.extend(lines)
            else:
                text = dd_tag.get_text(separator="\n", strip=True).replace('\xa0', ' ')
                lines = [line.strip() for line in text.split('\n') if line.strip()]
                combined_lines.extend(lines)

        data[key] = combined_lines if len(combined_lines) > 1 else combined_lines[0] if combined_lines else ""

    # 누락된 항목 기본값 추가
    for key in target_keys:
        if key not in data:
            data[key] = ""

    return data

# ✅ 크롤링 시작
start_time = time.time()
all_products = []
current_page = 1
max_page = 72
success_count = 0
fail_count = 0

while current_page <= max_page:
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    items = soup.find_all(class_="onOffTr")

    for idx, item in enumerate(items):
        lines = [line.strip() for line in item.text.strip().split('\n') if line.strip()]
        if len(lines) < 8:
            print(f"⚠️ {current_page}페이지 - {idx+1}번 상품: 기본 정보 부족으로 스킵")
            fail_count += 1
            continue

        base_info = lines[:8]

        try:
            detail_buttons = driver.find_elements(By.CSS_SELECTOR, 'a.tabel-more')
            driver.execute_script("arguments[0].click();", detail_buttons[idx])
            time.sleep(1.5)

            detail_soup = BeautifulSoup(driver.page_source, "html.parser")
            detailed_info = extract_detailed_info(detail_soup)

            product = {
                "금융회사": base_info[0],
                "상품명": base_info[1],
                "자금용도": base_info[2],
                "가입대상": base_info[3],
                "대출종류": base_info[4],
                "금리방식": base_info[5],
                "상환방식": base_info[6],
                "전월 평균금리": base_info[7],
                **detailed_info
            }

            all_products.append(product)
            success_count += 1

        except Exception as e:
            print(f"❌ {current_page}페이지 - {idx+1}번 상품: 상세 클릭 실패 → {e}")
            fail_count += 1
    print(f"✅ {current_page}페이지 수집 완료")

    # 다음 페이지 이동
    if current_page < max_page:
        try:
            next_page = current_page + 1
            next_btn = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, f'a[data-pageindex="{next_page}"]'))
            )
            driver.execute_script("arguments[0].click();", next_btn)
            current_page += 1
            time.sleep(2)
        except Exception as e:
            print(f"❌ {next_page}페이지 이동 실패: {e}")
            break
    else:
        break

# ✅ JSON 저장
with open("금융상품한눈에.json", "w", encoding="utf-8") as f:
    json.dump(all_products, f, ensure_ascii=False, indent=2)

end_time = time.time()
print(f"\n📦 총 상품 수집 성공: {success_count}개")
print(f"⚠️ 실패 또는 누락 상품: {fail_count}개")
print("📁 금융상품 데이터를 JSON 형식으로 저장 완료!")
print(f"\n⏱️ 총 소요 시간: {end_time - start_time:.2f}초")

driver.quit()


✅ 1페이지 수집 완료
✅ 2페이지 수집 완료
✅ 3페이지 수집 완료
✅ 4페이지 수집 완료
✅ 5페이지 수집 완료
✅ 6페이지 수집 완료
✅ 7페이지 수집 완료
✅ 8페이지 수집 완료
✅ 9페이지 수집 완료
✅ 10페이지 수집 완료
✅ 11페이지 수집 완료
✅ 12페이지 수집 완료
✅ 13페이지 수집 완료
✅ 14페이지 수집 완료
✅ 15페이지 수집 완료
✅ 16페이지 수집 완료
✅ 17페이지 수집 완료
✅ 18페이지 수집 완료
✅ 19페이지 수집 완료
✅ 20페이지 수집 완료
✅ 21페이지 수집 완료
✅ 22페이지 수집 완료
✅ 23페이지 수집 완료
✅ 24페이지 수집 완료
✅ 25페이지 수집 완료
✅ 26페이지 수집 완료
✅ 27페이지 수집 완료
✅ 28페이지 수집 완료
✅ 29페이지 수집 완료
✅ 30페이지 수집 완료
✅ 31페이지 수집 완료
✅ 32페이지 수집 완료
✅ 33페이지 수집 완료
✅ 34페이지 수집 완료
✅ 35페이지 수집 완료
✅ 36페이지 수집 완료
✅ 37페이지 수집 완료
✅ 38페이지 수집 완료
✅ 39페이지 수집 완료
✅ 40페이지 수집 완료
✅ 41페이지 수집 완료
✅ 42페이지 수집 완료
✅ 43페이지 수집 완료
✅ 44페이지 수집 완료
✅ 45페이지 수집 완료
✅ 46페이지 수집 완료
✅ 47페이지 수집 완료
✅ 48페이지 수집 완료
✅ 49페이지 수집 완료
✅ 50페이지 수집 완료
✅ 51페이지 수집 완료
✅ 52페이지 수집 완료
✅ 53페이지 수집 완료
✅ 54페이지 수집 완료
✅ 55페이지 수집 완료
✅ 56페이지 수집 완료
✅ 57페이지 수집 완료
✅ 58페이지 수집 완료
✅ 59페이지 수집 완료
✅ 60페이지 수집 완료
✅ 61페이지 수집 완료
✅ 62페이지 수집 완료
✅ 63페이지 수집 완료
✅ 64페이지 수집 완료
✅ 65페이지 수집 완료
✅ 66페이지 수집 완료
✅ 67페이지 수집 완료
✅ 68페이지 수집 완료
✅ 69페이지 수집 완료
✅ 70페이지 수집 완료
✅ 71페이지 수집 완료
✅ 72페이지 수집 완료



In [46]:
from collections import OrderedDict
import json

# 저장한 파일 경로
file_path = "금융상품한눈에.json"

with open("금융상품한눈에.json", "r", encoding="utf-8") as f:
    data = json.load(f, object_pairs_hook=OrderedDict)

# 개수 확인
print(f"총 상품 개수: {len(data)}개")

# 첫 번째 상품 미리보기
import pprint
pprint.pprint(data[1], width=120)


총 상품 개수: 10개
OrderedDict([('금융회사', '농협은행주식회사'),
             ('상품명', '채움 상생론'),
             ('자금용도', '일반'),
             ('가입대상', '제한없음'),
             ('대출종류', '담보대출, 보증대출, 신용대출'),
             ('금리방식', '변동금리'),
             ('상환방식', '만기일시상환, 원금분할상환, 원리금분할상환'),
             ('전월 평균금리', '2.58%'),
             ('적용가능 금리', ['최고금리: 5.04%', '최저금리: 5.04%']),
             ('대출한도', '개별협약 내용에 따름'),
             ('가입대상 세부요건', 'NH농협은행과 상생펀드 구성 협약을 체결한 대기업의 협력 중소기업(개인사업자 포함)'),
             ('가입방법', '영업점'),
             ('우대금리', '-'),
             ('중도상환 수수료',
              ['· 중도상환해약금 = [중도상환원금×적용요율×(잔여기간÷대출기간)]',
               '※ 중도상환해약금 적용 요율',
               '[고정금리대출인 경우]',
               '- 기업대출(부동산담보) : 1.4%',
               '- 기업대출(신용/보증서/기타담보) : 1.1%',
               '[변동금리대출인 경우]',
               '- 기업대출(부동산담보) : 1.2%',
               '- 기업대출(신용/보증서/기타담보) : 1.0%',
               '※ 대출취급일로부터 3년까지 적용합니다.']),
             ('대출부대비용',
              ['· 인지세(공통)',
               '인지세법에 의해 대출약

# JSON → 자연어 서술형 변환 코드

### 리스트 있는 형식

In [56]:
import json
import os
import re
from typing import Union, List

def is_missing(value: Union[str, List]) -> bool:
    if value is None:
        return True
    if isinstance(value, str) and value.strip() == "-":
        return True
    return False

def sanitize_filename(s: str) -> str:
    s = re.sub(r"[^\w가-힣]", "", s)
    return s[:30]

def convert_loan_json_to_text(data):
    lines = []

    lines.append(f"{data['금융회사']}에서는 '{data['상품명']}' 상품을 제공합니다.")
    lines.append(f"자금용도는 {data['자금용도']}입니다.")
    lines.append(f"대출 종류는 {data['대출종류']}입니다.")
    lines.append(f"금리 방식은 {data['금리방식']}입니다.")
    lines.append(f"상환 방식은 {data['상환방식']}입니다.")
    lines.append(f"가입 대상은 {data['가입대상']}입니다.")

    if not is_missing(data.get("가입대상 세부요건")):
        lines.append(f"구체적인 가입 요건은 '{data['가입대상 세부요건']}'입니다.")

    if not is_missing(data.get("전월 평균금리")):
        lines.append(f"전월 평균금리는 {data['전월 평균금리']}입니다.")
    else:
        lines.append("전월 평균금리는 제공되지 않았습니다.")

    if isinstance(data.get("적용가능 금리"), list) and len(data["적용가능 금리"]) == 2:
        최고금리 = data["적용가능 금리"][0].split(": ")[1]
        최저금리 = data["적용가능 금리"][1].split(": ")[1]
        lines.append(f"적용 가능 금리는 최고 {최고금리}, 최저 {최저금리}입니다.")

    if not is_missing(data.get("대출한도")):
        lines.append(f"대출 한도는 {data['대출한도']}입니다.")

    if not is_missing(data.get("우대금리")):
        lines.append(f"우대금리는 {data['우대금리']}입니다.")

    if not is_missing(data.get("가입방법")):
        lines.append(f"가입은 {data['가입방법']}을 통해 가능합니다.")

    수수료 = data.get("중도상환 수수료")
    if 수수료 and not is_missing(수수료):
        if isinstance(수수료, list):
            lines.append("중도상환 수수료:")
            lines.extend([f"- {s.strip('-').strip()}" for s in 수수료])
        else:
            lines.append(f"중도상환 수수료는 {수수료}입니다.")

    비용 = data.get("대출부대비용")
    if 비용 and not is_missing(비용):
        if isinstance(비용, list):
            lines.append("대출 부대비용:")
            lines.extend([f"- {b.strip('-').strip()}" for b in 비용])
        else:
            lines.append(f"대출 부대비용에는 {비용} 등이 포함됩니다.")

    연체 = data.get("연체이자율")
    if 연체 and not is_missing(연체):
        if isinstance(연체, list):
            lines.append("연체 이자율:")
            lines.extend([f"- {r}" for r in 연체])
        else:
            lines.append(f"연체 이자율은 {연체}입니다.")

    담당 = data.get("담당부서 및 연락처")
    if 담당 and not is_missing(담당):
        if isinstance(담당, list):
            lines.append("담당 부서 및 연락처:")
            lines.extend([f"- {d}" for d in 담당])
        else:
            lines.append(f"담당 부서 및 연락처는 {담당}입니다.")

    return "\n".join(lines)

def process_json_to_text(input_json_path, output_dir):
    with open(input_json_path, "r", encoding="utf-8") as f:
        items = json.load(f)

    os.makedirs(output_dir, exist_ok=True)

    for idx, item in enumerate(items, start=1):
        금융회사 = sanitize_filename(item.get("금융회사", "unknown"))
        상품명 = sanitize_filename(item.get("상품명", f"상품_{idx}"))

        filename = f"{str(idx).zfill(3)}_{금융회사}_{상품명}.txt"
        filepath = os.path.join(output_dir, filename)

        text = convert_loan_json_to_text(item)

        with open(filepath, "w", encoding="utf-8") as out:
            out.write(text)

        print(f"✅ 저장 완료: {filename}")

# 실행 예시
process_json_to_text("금융상품한눈에.json", "금융상품_chunks")

✅ 저장 완료: 001_국민은행_KB동반성장협약상생대출.txt
✅ 저장 완료: 002_농협은행주식회사_채움상생론.txt
✅ 저장 완료: 003_우리은행_우리CUBE론XBIZ프라임.txt
✅ 저장 완료: 004_중소기업은행_IBK일자리Plus대출.txt
✅ 저장 완료: 005_농협은행주식회사_NH산업단지대출.txt
✅ 저장 완료: 006_주식회사케이뱅크_사장님부동산담보대출.txt
✅ 저장 완료: 007_농협은행주식회사_행복채움프랜차이즈론.txt
✅ 저장 완료: 008_중소기업은행_중기부RD사업화자금대출.txt
✅ 저장 완료: 009_우리은행_우리CUBE론X우리자산신탁.txt
✅ 저장 완료: 010_국민은행_KB우량산업단지기업우대대출.txt
✅ 저장 완료: 011_중소기업은행_IBK스마트공장지원대출.txt
✅ 저장 완료: 012_중소기업은행_IBK시설투자대출.txt
✅ 저장 완료: 013_중소기업은행_동반성장협력대출.txt
✅ 저장 완료: 014_중소기업은행_ESG경영성공지원시설자금대출.txt
✅ 저장 완료: 015_한국스탠다드차타드은행_비즈니스모기지Mortgage.txt
✅ 저장 완료: 016_국민은행_KB더드림TheDream소호대출.txt
✅ 저장 완료: 017_국민은행_KB유망분야성장기업우대대출.txt
✅ 저장 완료: 018_농협은행주식회사_중소기업보증료지원대출.txt
✅ 저장 완료: 019_중소기업은행_IBK혁신창업대출.txt
✅ 저장 완료: 020_국민은행_KB일사천리소호대출.txt
✅ 저장 완료: 021_국민은행_기업일반시설자금대출.txt
✅ 저장 완료: 022_중소기업은행_첨단기술기업육성자금대출.txt
✅ 저장 완료: 023_중소기업은행_IBK문화콘텐츠대출.txt
✅ 저장 완료: 024_우리은행_우리CUBE론X일반산업단지.txt
✅ 저장 완료: 025_농협은행주식회사_NH기업성장론_착한임대인우대임대사업자전용.txt
✅ 저장 완료: 026_수협은행_Sh전문직사업자파트너론.txt
✅ 저장 완료: 027_아이엠뱅크_무브온MoveOn특별대출.txt
✅ 저장 

### 리스트 없이 줄글 형식

In [ ]:
import json
import os
import re
from typing import Union, List

# 값 누락 확인
def is_missing(value: Union[str, List]) -> bool:
    if value is None:
        return True
    if isinstance(value, str) and value.strip() == "-":
        return True
    return False

# 파일명 안전하게 정리
def sanitize_filename(s: str) -> str:
    s = re.sub(r"[^\w가-힣]", "", s)
    return s[:30]

# 대출부대비용 등 일반 리스트 → 문장 처리
def format_list_to_sentences(items: List[str]) -> List[str]:
    formatted = []
    for item in items:
        item = item.lstrip("-").strip()
        if ":" in item:
            key, value = item.split(":", 1)
            sentence = f"{key.strip()}는 {value.strip()}"
        else:
            sentence = item

        if sentence.endswith("니다.") or sentence.endswith("다."):
            formatted.append(sentence)
        else:
            formatted.append(sentence + "입니다.")
    return formatted

# 중도상환 수수료 특수 처리
def format_prepayment_fee_sentences(items: List[str]) -> List[str]:
    result = []
    for item in items:
        item = item.strip()
        if any(op in item for op in ["X", "x", "÷", "*", "="]):
            result.append(f'"{item}"으로 계산됩니다.')
        elif "최장" in item or "발생" in item:
            result.append(item if item.endswith("다.") else item + "합니다.")
        elif "수수료율" in item:
            result.append("수수료율은 다음과 같습니다.")
        elif item.startswith("-"):
            try:
                title, value = item.lstrip("-").strip().split(":", 1)
                result.append(f"{title.strip()}의 수수료율은 {value.strip()}입니다.")
            except ValueError:
                sentence = item.lstrip("-").strip()
                result.append(sentence if sentence.endswith("니다.") or sentence.endswith("다.") else sentence + "입니다.")
        else:
            result.append(item if item.endswith("니다.") or item.endswith("다.") else item + "입니다.")
    return result

# 전체 상품 정보를 문장으로 변환
def convert_loan_json_to_text(data):
    lines = [f"{data['금융회사']}에서 '{data['상품명']}' 상품을 제공합니다."]
    lines.append(f"자금용도는 {data['자금용도']}입니다.")
    lines.append(f"대출 종류는 {data['대출종류']}입니다.")
    lines.append(f"금리 방식은 {data['금리방식']}입니다.")
    lines.append(f"상환 방식은 {data['상환방식']}입니다.")
    lines.append(f"가입 대상은 {data['가입대상']}입니다.")

    if not is_missing(data.get("가입대상 세부요건")):
        lines.append(f"구체적인 가입 요건은 '{data['가입대상 세부요건']}'입니다.")

    if not is_missing(data.get("전월 평균금리")):
        lines.append(f"전월 평균금리는 {data['전월 평균금리']}입니다.")
    else:
        lines.append("전월 평균금리는 제공되지 않았습니다.")

    if isinstance(data.get("적용가능 금리"), list) and len(data["적용가능 금리"]) == 2:
        최고금리 = data["적용가능 금리"][0].split(": ")[1]
        최저금리 = data["적용가능 금리"][1].split(": ")[1]
        lines.append(f"적용 가능 금리는 최고 {최고금리}, 최저 {최저금리}입니다.")

    if not is_missing(data.get("대출한도")):
        lines.append(f"대출 한도는 {data['대출한도']}입니다.")

    if not is_missing(data.get("우대금리")):
        lines.append(f"우대금리는 {data['우대금리']}입니다.")

    if not is_missing(data.get("가입방법")):
        lines.append(f"가입은 {data['가입방법']}을 통해 가능합니다.")

    수수료 = data.get("중도상환 수수료")
    if 수수료 and not is_missing(수수료):
        lines.append("중도상환 수수료는 다음과 같습니다.")
        if isinstance(수수료, list):
            lines.extend(format_prepayment_fee_sentences(수수료))
        else:
            lines.append(f"{수수료}입니다.")

    비용 = data.get("대출부대비용")
    if 비용 and not is_missing(비용):
        lines.append("대출 부대비용은 다음과 같습니다.")
        if isinstance(비용, list):
            lines.extend(format_list_to_sentences(비용))
        else:
            lines.append(f"{비용} 등이 포함됩니다.")

    연체 = data.get("연체이자율")
    if 연체 and not is_missing(연체):
        if isinstance(연체, list):
            lines.append(f"연체 이자율은 {' '.join(연체)}입니다.")
        else:
            lines.append(f"연체 이자율은 {연체}입니다.")

    담당 = data.get("담당부서 및 연락처")
    if 담당 and not is_missing(담당):
        lines.append("담당 부서 및 연락처는 다음과 같습니다.")
        if isinstance(담당, list):
            for d in 담당:
                d = d.strip()
                if d.endswith("니다.") or d.endswith("다."):
                    lines.append(d)
                else:
                    lines.append(f"{d}입니다.")
        else:
            d = 담당.strip()
            if d.endswith("니다.") or d.endswith("다."):
                lines.append(d)
            else:
                lines.append(f"{d}입니다.")

    return "\n".join(lines)

# 실행 함수
def process_json_to_text(input_json_path, output_dir):
    with open(input_json_path, "r", encoding="utf-8") as f:
        items = json.load(f)

    os.makedirs(output_dir, exist_ok=True)

    for idx, item in enumerate(items, start=1):
        금융회사 = sanitize_filename(item.get("금융회사", "unknown"))
        상품명 = sanitize_filename(item.get("상품명", f"상품_{idx}"))
        filename = f"{str(idx).zfill(3)}_{금융회사}_{상품명}.txt"
        filepath = os.path.join(output_dir, filename)

        text = convert_loan_json_to_text(item)

        with open(filepath, "w", encoding="utf-8") as out:
            out.write(text)

        print(f"✅ 저장 완료: {filename}")

# 실행 예시
process_json_to_text("금융상품한눈에.json", "금융상품_chunks")


✅ 저장 완료: 001_국민은행_KB동반성장협약상생대출.txt
✅ 저장 완료: 002_농협은행주식회사_채움상생론.txt
✅ 저장 완료: 003_우리은행_우리CUBE론XBIZ프라임.txt
✅ 저장 완료: 004_중소기업은행_IBK일자리Plus대출.txt
✅ 저장 완료: 005_농협은행주식회사_NH산업단지대출.txt
✅ 저장 완료: 006_주식회사케이뱅크_사장님부동산담보대출.txt
✅ 저장 완료: 007_농협은행주식회사_행복채움프랜차이즈론.txt
✅ 저장 완료: 008_중소기업은행_중기부RD사업화자금대출.txt
✅ 저장 완료: 009_우리은행_우리CUBE론X우리자산신탁.txt
✅ 저장 완료: 010_국민은행_KB우량산업단지기업우대대출.txt
✅ 저장 완료: 011_중소기업은행_IBK스마트공장지원대출.txt
✅ 저장 완료: 012_중소기업은행_IBK시설투자대출.txt
✅ 저장 완료: 013_중소기업은행_동반성장협력대출.txt
✅ 저장 완료: 014_중소기업은행_ESG경영성공지원시설자금대출.txt
✅ 저장 완료: 015_한국스탠다드차타드은행_비즈니스모기지Mortgage.txt
✅ 저장 완료: 016_국민은행_KB더드림TheDream소호대출.txt
✅ 저장 완료: 017_국민은행_KB유망분야성장기업우대대출.txt
✅ 저장 완료: 018_농협은행주식회사_중소기업보증료지원대출.txt
✅ 저장 완료: 019_중소기업은행_IBK혁신창업대출.txt
✅ 저장 완료: 020_국민은행_KB일사천리소호대출.txt
✅ 저장 완료: 021_국민은행_기업일반시설자금대출.txt
✅ 저장 완료: 022_중소기업은행_첨단기술기업육성자금대출.txt
✅ 저장 완료: 023_중소기업은행_IBK문화콘텐츠대출.txt
✅ 저장 완료: 024_우리은행_우리CUBE론X일반산업단지.txt
✅ 저장 완료: 025_농협은행주식회사_NH기업성장론_착한임대인우대임대사업자전용.txt
✅ 저장 완료: 026_수협은행_Sh전문직사업자파트너론.txt
✅ 저장 완료: 027_아이엠뱅크_무브온MoveOn특별대출.txt
✅ 저장 

## png에서 텍스트 추출 

In [7]:
pip install pytesseract pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from PIL import Image
import pytesseract

# (Windows 사용 시 주석 해제 후 Tesseract 경로 지정)
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# 이미지 열기
image_path = "공고문.png"  # 업로드한 이미지 경로
image = Image.open(image_path)

# 텍스트 추출
text = pytesseract.image_to_string(image, lang='kor+eng')  # 한글과 영어 동시 인식

# 결과 출력
print(text)


20258 SSA AAl SSS este w=

es AAW

O MASA: 4S Wl 74 aS 2 ae Se} MEY Soe
AGL Aat Sk AEMAlel Pus oppaal alee Fal
FEBS 7A baal 710}
* BE SESAME SCHARAD 1982('25.5.30,)
O Ws: su Asdas opoe Asaa Fo A e2hlal
- 100% 21 FRE] Ola AQ] Og ask HShipollAy ol7} apo] gl

ac =R2ue

Oana

O AFAAS FU 7ASASS BAe Su V4

O ASLAS +UGeupste] ASS aAzAAbs] Bar Eb alo)
STE HE SS MM SEL ASHE UAE Aly]

O ASS: 254 SSA ASS FB AAG SQ Hay)

LC) AMAA: 84 150098 tHe, Wesel, 14 clu] YA) Ast
+ CHEALO| SS7AKSe TALCH)O| DASH Bao] 1% ALS BeUEWIE 2.99%) HS
+ YD 32 HS ABAM BRLANS YA] Se HO SAA
» S712: NHSwSse

O WSs

O Bar AS 3azi atts AHS) Seat 9H 1~12.3L “See FOB 7}4))-2-
WA) He WI Bh 414, aE RE oy 21d
+ 32 342! AAO] SE AS, ADAG Be sQH WBOe HE AS
~ SBS AIHA GlA| SSDI, ASAE Soll We} HERTZ 2

C) AKSSEE: aid AHO] Ait el SS 7 al SPO URES: ORE ah

« a2 ABAM FAS At HSS ABST SE CHO] PISS, TIS BAAS WS

Ae B55 S7}
OF SIMCHA A9H)7I/Z#: ° 25. 1. 1. (42) ~7 25. 12. 31. >)
O tS At

E

Ed

ALUAL 4 As SM

Ciel Be AS)
Abe CA Sues aa eae

a

In [39]:
import cv2
import numpy as np
from PIL import Image
import pytesseract

# ✅ Tesseract 경로 설정
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# ✅ 이미지 경로
image_path = 'image.png'

# ✅ 이미지 불러오기 및 전처리
img = cv2.imread(image_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
_, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

# 예시: 일정 크기 이하의 정사각형 검정 기호 제거 (단순한 예)
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
for cnt in contours:
    x, y, w, h = cv2.boundingRect(cnt)
    if w < 15 and h < 15:  # 네모 크기 기준
        cv2.rectangle(thresh, (x, y), (x+w, y+h), 255, -1)  # 흰색으로 덮기


# 이미지 확대
# scale_percent = 150  # 150% 확대
# width = int(thresh.shape[1] * scale_percent / 100)
# height = int(thresh.shape[0] * scale_percent / 100)
# resized = cv2.resize(thresh, (width, height), interpolation=cv2.INTER_CUBIC)

# ✅ OCR 수행
custom_config = r'--psm 6'  # 한 줄씩 읽는 구조

text = pytesseract.image_to_string(thresh, lang='kor', config=custom_config)


# ✅ 출력
print(text)



2025 년 식 품 소 재 업 계 경 영 안 정 자 금 지 원 안 내 문
매 사 이 개 요
니 사 업 목 적 : 식 품 원 자 재 가 격 상 승 및 환 율 글 리 ㆍ 유 가 변 동 성 등 으 로
경 영 난 을 겪 고 있 는 식 품 업 계 에 구 매 자 금 이 자 보 전 지 원 을 통 해
가 옹 식 품 가 격 안 정 에 기 어
* 관 련 : 농 림 축 산 식 품 부 푸 드 테 크 정 책 과 -1989(25.5.30.)

디 지 원 내 용 : 국 내 식 품 업 체 를 대 상 으 로 식 품 소 재 구 매 자 금 융 자 지 원
- 1500 익 원 규 모 의 이 차 보 전 사 엄 으 로 예 산 멈 위 내 에 서 이 자 자 액 지 원
때 주 489

니 지 원 대상
2 식 줌 소 재 률 수 입 하 여 가 공 식 품 을 생 산 하 는 국 내 업 체
ㅇ 식 품 소 재 를 수 입 ( 구 배 하 여 식 품 을 재 조 . 생 산 하 지 않 고 다 업 체 에 게
공 급 하 는 유 통 능 을 위 한 용 도 로 사 용 하 는 입 체 는 제 외
니 지 원 픔 목 : '25 년 한 당 관 세 지 원 품 목 및 유 지 류 등 ( 불 임1 참 고 )
디 치 원 조 건 : 융 자 1.500 억 원 규 모 , 변 동 금 리 , 1 년 이 내 일 시 상 환
* 다 출 시 점 에 금 융 기 괜 농 협 가 계 담 보 대 출 ! 이 고 사 하 는 금 라 에 1% 틀 차 슴 한 금 리 (6 돌 기 준 3.33%; 적 룡
* 최 근 3 년 간 해 당 식 품 소 재 평 균 수 입 액 을 넘 지 않 는 범 위 에 서 융 자 지 원
+ 대 출 기 관 : 4+{ 농 협 은 행
니 지 원 한 도
ㅇ 임 체 명 과근 3 간 해 당 식 품 4 빼 평 균 수 임 액 11~1231. 동 관 기 준 808 가 격 ) 을
넘 지 않 는 법 위 에 서 융 자 신 청 , 예 산 한 도 내 에 서 지 원
* 최 근 3 년 간 실 적 이 없 는 겸 후 , 최 근 연 도 평 균 수 입 액 기 준 므 로 한 도 산 출
＊ 융 자 금 온

In [21]:
with open('image.output.txt', 'w', encoding='utf-8') as f:
    f.write(text)


In [32]:
import cv2
import pytesseract
from PIL import Image
import numpy as np

# 경로 설정
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
image_path = "image.png"  

# 이미지 불러오기
img = cv2.imread(image_path)

# 전처리: 흑백 변환 + adaptive thresholding
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray = cv2.GaussianBlur(gray, (3, 3), 0)
thresh = cv2.adaptiveThreshold(
    gray, 255, 
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
    cv2.THRESH_BINARY, 11, 2
)

# 이미지 확대
scale_percent = 150  # 150% 확대
width = int(thresh.shape[1] * scale_percent / 100)
height = int(thresh.shape[0] * scale_percent / 100)
resized = cv2.resize(thresh, (width, height), interpolation=cv2.INTER_CUBIC)

# OCR 수행
custom_config = r'--oem 3 --psm 4'  # 문단 인식 중심
text = pytesseract.image_to_string(resized, lang='kor', config=custom_config)

print(text)


.2025 년 : 식 품 소 재 : 업 계 경 영 안 정 : 자 금 지 원 안 내 문.

| : |^자업개요
[ 사 업 복 적 : 식풍 원 자 체 가 격 : 상 승 및 환 율 큼 리 ㆍ 유 가 " 변 동 성 등 으 로
경 영 난 을 겪 고 . 있 는 식 품 업 계 에 구 매 자 금 : 이 차 보 전 지 원 을 통 해
가 공 식 품 가 격 안 정 에 기 여
ㅜ : 관 련 : 농 림 축 산 식 품 부 푸 드 테 크 정 책 과 -1982('25.5.80.)
[] 지 원 내 용 : 국 니 식 품 업 체 를 데 상 으 로 식 품 소제 구 매 자 금 융 자 지 원
- 1500 억 원 큐 모 의 이 차 보 전 : 사 엄 으 로 에 산 - 위 내 에 서 이 자 차 액 지 원

니 지 원 대상
으 식 품 소 제 를 . 수 입 하 여 . 가 공 식 품 을 ' 생 산 하 는 곡 내 업 체 .
ㅇ 식 품 소 재 를 수 입 (? 매 하 여 .: 식 품 율 제 조 - 생 산 하 지 ' 않 고 타 . 업 체 에 게
공 급 하 는 유 동 등 을 위 한 용 토 로 . 사 용 하 는 업 체 는 제 외
니 지 원 품 목 ::'25 년 할 당 관 세 지 원 품 목 및 : 유 지 류 ' 등 ( 붙 임 1、 참 고 )
디 지 원 조 건 : 융 자 1,500 억 원 규 모 , : 변 동 금 리 , 1 년 ' 이 내 일 시 상 환
* 대 충 시 점 에 금 융 기 관 농 협 기 게 덤 보 대 출 )이 고 시 하 는 금 리 에 1% 틀 치 감 한 금 리 (6 월 기 준 3339 적 용
: 최 근 .3 년 간 : 해 당 - 식 품 소 재 : 평 균 수 입 액 을 . 넘 지 : 않 는 - 법 위 에 서 용 자 지 원
…: 대 출 기 관 : 배 능 협 은 행
디 지 원 한 도 '
< ㅇ > 、 업 체 빌 최 근 3 년 간 해 당 식 품 소 재 : 평 균 수 입 액 1.1~1231. 동 관 기 준 .608- 가 격 ) 율
넘 지 않 는 범위에서 ~ 응 자 신 청 ,, 예 산 한 도 . 내 에 서 지 원
*: 최 